 - 参考链接：https://huggingface.co/learn/llm-course/chapter11/4

In [1]:
%env HF_ENDPOINT=https://hf-mirror.com
# %env HF_ENDPOINT=https://huggingface.co
%env HF_HOME=/root/autodl-tmp/hf

# LoRA 依赖 PEFT。安装完成后请 Kernel → Restart，再运行后续单元。
!pip install peft trl datasets

env: HF_ENDPOINT=https://hf-mirror.com
env: HF_HOME=/root/autodl-tmp/hf
Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
# 加载model和Tokenizer
from transformers import AutoModelForCausalLM,AutoTokenizer
import torch

model_name = 'Qwen/Qwen3-8B'
model = AutoModelForCausalLM.from_pretrained(model_name,dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [3]:
# 数据集
from datasets import load_dataset
dataset_dict = load_dataset('json',data_files={"train":"data/keywords_data_train.jsonl","test":"data/keywords_data_test.jsonl"})

def map_func(example):
    conversation = example['conversation']
    messages = []
    for item in conversation:
        messages.append({'role':'user','content':item['human']})
        messages.append({'role':'assistant','content':item['assistant']})
    return {'messages':messages}

dataset_dict = dataset_dict.map(map_func,batched=False,remove_columns=['dataset','conversation','category','conversation_id'])

In [4]:
# 流程：先运行上一格的 pip install，然后务必「重启内核」，再从上到下执行本 Notebook。
# 原因：若曾在未安装 peft 时 import 过 trl，用 importlib.reload(transformers...) 虽能临时修
# NameError，但会在保存 checkpoint 时触发 PicklingError（SchedulerType 等枚举不是同一对象）。
from trl import SFTConfig, SFTTrainer


In [5]:

from peft import LoraConfig

import os

# TensorBoard：Transformers 5.x 用 TENSORBOARD_LOGGING_DIR；logging_dir 参数已废弃且不会写入该路径
# 必须在实例化 SFTTrainer 之前设置，以便 TensorBoardCallback 读到
os.environ["TENSORBOARD_LOGGING_DIR"] = "/root/tf-logs"

# TODO: Configure LoRA parameters
# r: rank dimension for LoRA update matrices (smaller = more compression)
rank_dimension = 4
# lora_alpha: scaling factor for LoRA layers (higher = stronger adaptation)
lora_alpha = 8
# lora_dropout: dropout probability for LoRA layers (helps prevent overfitting)
lora_dropout = 0.05

peft_config = LoraConfig(
    r=rank_dimension,  # Rank dimension - typically between 4-32
    lora_alpha=lora_alpha,  # LoRA scaling factor - typically 2x rank
    lora_dropout=lora_dropout,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA. the corresponding biases will be updated during training.
    target_modules="all-linear",  # Which modules to apply LoRA to
    task_type="CAUSAL_LM",  # Task type for model architecture
)


# Configure trainer
training_args = SFTConfig(
    output_dir="/root/autodl-tmp/.autodl/sft/Qwen3-8B/sft-LoRA",
    max_steps=1000,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    logging_steps=10,
    report_to="tensorboard",
    save_steps=100,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=100,
    load_best_model_at_end=True,
    bf16=True,
    warmup_steps=50
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["test"],
    processing_class=tokenizer,
    peft_config=peft_config
)

In [6]:
# 察看数据集处理结果
dataloader = trainer.get_train_dataloader()
batch = next(iter(dataloader))
print(tokenizer.decode(batch['input_ids'][0]))

<|im_start|>user
关键词抽取：
以手动换挡机构疲劳寿命试验为目的,构建了一种模拟驾驶员进行选档、换挡操作的试验平台,该平台集成二自由度运动滑台与气动加载装置为一体形成换挡运动加载机构.以该机构为研究对象,通过建立换挡与选挡的运动轨迹模型,分析换挡运动加载机构位移输出与运动轨迹之间的关系,通过分析换挡机构操纵杆受力情况,分别对换挡动作和选档动作进行力学分析,并得出加载力的计算方法.最后结合电气控制技术与气动控制技术,对系统进行了试验,结果表明,系统具有可行性与正确性.<|im_end|>
<|im_start|>assistant
<think>

</think>

换挡机构;试验平台;疲劳性能<|im_end|>
<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


In [7]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
100,2.249719,2.125962
200,1.885942,1.990717
300,2.143629,1.975304
400,2.009493,1.969066
500,2.073315,1.962994
600,1.914125,1.960155
700,2.091246,1.958257
800,2.005466,1.956507
900,2.013481,1.955187
1000,1.949654,1.954631


TrainOutput(global_step=1000, training_loss=2.1125349979400636, metrics={'train_runtime': 663.3427, 'train_samples_per_second': 6.03, 'train_steps_per_second': 1.508, 'total_flos': 4.81544342034432e+16, 'train_loss': 2.1125349979400636})

In [8]:
trainer.save_model('/root/autodl-tmp/.autodl/sft/Qwen3-8B/sft-LoRA/best')

In [9]:
next(model.parameters()).dtype

torch.bfloat16